# 6. Validación estadística

El modelo original se considera superior solo si (guía, sección 6.1): (1) su AUC medio del bucle
externo supera al de la mejor combinación de cada uno de los siete modelos, (2) al menos tres pruebas
distintas rechazan la igualdad de desempeño con p ajustado por comparaciones múltiples menor que 0,05
(0,10 en el MCS) y (3) el tamaño del efecto no es despreciable. Este capítulo aplica ese protocolo y
reporta también las pruebas que no rechazan.

## 6.1 Protocolo

* **Universo:** el AJ-KL y las siete mejores combinaciones del capítulo 3 (los benchmarks del curso), con las mismas particiones
  externas, la misma semilla y el mismo preprocesamiento. Las predicciones de los benchmarks son las del
  Entregable 2, emparejadas fila por fila con las del AJ-KL. Los dos comparadores recientes del estado del arte
  (LightGBM y CatBoost, ajustados con el mismo protocolo) se contrastan aparte en la sección 6.8.
* **Bloques:** el bucle externo cubre cuatro años (k1-k4) y se divide en 48 bloques mensuales; la prueba
  2025 tiene 52 semanas. Usar meses y semanas en lugar de solo cinco pliegues da potencia a las pruebas
  no paramétricas sin repetir la búsqueda de hiperparámetros, que tardó 29 horas-CPU para los 140 modelos.
  Los meses de un mismo pliegue comparten modelo y año, así que no son independientes: la prueba de
  Friedman se complementa con una permutación restringida al pliegue y las pruebas sobre 2025 usan errores
  estándar por bloques de semana o bootstrap estacionario.
* **Ajuste por comparaciones múltiples:** Holm dentro de cada familia de siete comparaciones (AJ-KL frente
  a cada benchmark); Nemenyi ya controla el error de familia de sus 28 pares; el MCS controla el error de
  familia por construcción.

### Fundamento estadístico de las pruebas

Sean $k$ modelos comparados en $N$ bloques (los 48 meses del bucle externo) y $d_b$ la diferencia de una
métrica entre el modelo original y un benchmark en el bloque $b$. Cada prueba responde una pregunta distinta y
trata la dependencia entre bloques de una forma distinta; por eso la guía pide al menos tres.

**Friedman** {cite}`friedman1937`. Con $R_j$ el rango medio del modelo $j$ en los $N$ bloques,

$$
\chi^2_F = \frac{12N}{k(k+1)}\Big[\sum_{j=1}^k R_j^2 - \frac{k(k+1)^2}{4}\Big],
\qquad
F_F = \frac{(N-1)\,\chi^2_F}{N(k-1) - \chi^2_F},
$$ (eq-friedman)

con $F_F \sim F_{k-1,\,(k-1)(N-1)}$ bajo $H_0$: todos los modelos tienen la misma distribución de rangos
{cite}`iman1980`. La distribución de referencia supone bloques independientes; como los meses de un mismo
pliegue comparten modelo entrenado, el p-valor que se reporta es el de una **permutación restringida**: en
cada réplica se permutan las etiquetas de los modelos con la misma permutación en todos los meses de un
pliegue, y $p$ es la fracción de réplicas con estadístico mayor o igual que el observado.

**Nemenyi** {cite}`nemenyi1963,demsar2006`. Dos modelos difieren si $|R_i - R_j| > \mathrm{CD} =
q_\alpha\sqrt{k(k+1)/(6N)}$, con $q_\alpha$ el cuantil del rango studentizado dividido por $\sqrt 2$. Controla el
error de familia de todos los pares y por eso es conservadora.

**Wilcoxon de rangos con signo** {cite}`wilcoxon1945`. $W^+ = \sum_{b:\,d_b > 0}\mathrm{rango}(|d_b|)$; bajo
$H_0$ (las $d_b$ son simétricas alrededor de cero),
$z = \{W^+ - N(N+1)/4\} / \sqrt{N(N+1)(2N+1)/24}$ es aproximadamente normal. No supone normalidad, pero sí
simetría e independencia entre meses (la sección 6.3 revisa ambas). Los p-valores se ajustan con Holm
{cite}`holm1979`: ordenados $p_{(1)} \le \dots \le p_{(m)}$, $\tilde p_{(i)} = \max_{j \le i}\min\{1, (m-j+1)\,p_{(j)}\}$.

**DeLong** {cite}`delong1988,sun2014fast`. El AUC es un estadístico U,
$\hat\theta = \frac{1}{n_1 n_0}\sum_{i \in \mathcal P}\sum_{j \in \mathcal N}\psi(s_i, s_j)$ con
$\psi(a, b) = \mathbf 1[a > b] + \tfrac12\mathbf 1[a = b]$; sus componentes estructurales dan la covarianza de
$(\hat\theta_1, \hat\theta_2)$ en la misma muestra y $z = (\hat\theta_1 - \hat\theta_2)/\sqrt{\widehat{\mathrm{Var}}(\hat\theta_1 - \hat\theta_2)}$.
Esa varianza supone filas independientes; en 2025 las filas de una misma semana comparten choques, así que el
error estándar se reemplaza por el del **bootstrap por bloques de semana** y el intervalo es BCa
{cite}`efron1987bca`.

**Diebold-Mariano** {cite}`diebold1995` con la corrección de Harvey, Leybourne y Newbold {cite}`harvey1997`.
Con $d_t$ la diferencia de log-loss semanal entre el benchmark y el original en las $T = 52$ semanas de 2025,

$$
\mathrm{DM} = \frac{\bar d}{\sqrt{\hat\sigma^2_{\mathrm{HAC}}/T}},
\qquad
\mathrm{DM}^\ast = \mathrm{DM}\,\sqrt{\frac{T + 1 - 2h + h(h-1)/T}{T}} \sim t_{T-1},
$$ (eq-dm)

donde $\hat\sigma^2_{\mathrm{HAC}}$ es la varianza de largo plazo de Newey-West {cite}`newey1987hac`, que admite
autocorrelación de las diferencias, y $h = 1$ es el horizonte. Supone que $d_t$ es estacionaria en covarianza.

**Model Confidence Set** {cite}`hansen2011mcs`. Elimina modelos de uno en uno mientras la prueba de igual
capacidad predictiva (estadístico de rango sobre las diferencias de pérdida por pares) rechace; el conjunto que
queda contiene al mejor modelo con probabilidad al menos $1 - \alpha$. Las distribuciones se obtienen con el
bootstrap estacionario {cite}`politis1994`, que respeta la dependencia temporal.

**t corregida de Nadeau y Bengio** {cite}`nadeau2003`. Con $J$ pliegues,
$t = \bar d / \sqrt{(1/J + n_{\mathrm{eval}}/n_{\mathrm{entr}})\,s^2} \sim t_{J-1}$: el término
$n_{\mathrm{eval}}/n_{\mathrm{entr}}$ corrige la subestimación de la varianza por el solapamiento de los
entrenamientos. Con solo cinco pliegues tiene poca potencia, y por eso es complementaria.

**Tamaños del efecto.** Delta de Cliff $\delta = P(X > Y) - P(X < Y)$ {cite}`cliff1993` con los umbrales de
{cite:t}`romano2006cliff` (despreciable si $|\delta| < 0{,}147$) y $d_z = \bar d / s_d$ pareada. La regla de
decisión de la guía exige tres pruebas con $\tilde p < 0{,}05$ (0,10 en el MCS) **y** un efecto no despreciable.

| Prueba | $H_0$ | Distribución de referencia | Cómo trata la dependencia |
|---|---|---|---|
| Friedman | Mismos rangos para todos los modelos | $F_{k-1,(k-1)(N-1)}$ y permutación | Permutación restringida al pliegue |
| Nemenyi | Mismo rango medio por par | Rango studentizado | Ninguna (conservadora) |
| Wilcoxon-Holm | Mediana de $d_b$ igual a cero | Normal (aprox.) | Se revisa la autocorrelación (6.3) |
| DeLong | Mismo AUC en 2025 | Normal | Error estándar por bloques de semana |
| Diebold-Mariano HLN | Misma log-loss esperada | $t_{T-1}$ | Varianza HAC de Newey-West |
| MCS | Igual capacidad predictiva | Bootstrap | Bootstrap estacionario |
| Nadeau-Bengio | Misma métrica esperada | $t_{J-1}$ | Corrección de varianza por solapamiento |

In [1]:
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display
from scipy import stats

sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
pio.renderers.default = "notebook_connected"
import plots as PL

V = "../data/resultados/validacion/"
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
tabla = pd.read_parquet(V + "tabla_pruebas.parquet")
W = pd.read_parquet(V + "wilcoxon_mensual.parquet")
Dl = pd.read_parquet(V + "delong_2025.parquet")
DM = pd.read_parquet(V + "diebold_mariano_2025.parquet")
NB = pd.read_parquet(V + "nadeau_bengio.parquet")
res = json.loads(open(V + "resumen.json").read())
mens = pd.read_parquet(V + "metricas_mensuales_externo.parquet")
corto = lambda c: c.split(" | ")[0]

## 6.2 Criterio 1: AUC medio del bucle externo

In [2]:
sel = pd.read_csv("../data/resultados/seleccion_mejores.csv")
ejul = [joblib.load(f"../data/resultados/ajkl/{pl}.joblib")["resumen"] for pl in ["k1", "k2", "k3", "k4"]]
sota = {m: [joblib.load(f"../data/resultados/sota/{m}_{pl}.joblib")["resumen"] for pl in ["k1", "k2", "k3", "k4"]] for m in ["LightGBM", "CatBoost"]}
cols = ["auc", "auc_pr", "brier", "log_loss", "ece", "f1", "precision", "recall", "accuracy"]
comp = pd.DataFrame([{"modelo": "AJ-KL (original)", **{c: np.mean([e[c] for e in ejul]) for c in cols},
                      "auc_de": np.std([e["auc"] for e in ejul], ddof=1)}] +
                    [{"modelo": f"{m} (estado del arte)", **{c: np.mean([e[c] for e in v]) for c in cols},
                      "auc_de": np.std([e["auc"] for e in v], ddof=1)} for m, v in sota.items()] +
                    [{"modelo": r.modelo, **{c: getattr(r, f"{c}_media") for c in cols}, "auc_de": r.auc_de}
                     for r in sel.itertuples()]).set_index("modelo")
display(comp.round(4))
mejor = {c: (comp[c].idxmin() if c in ("brier", "log_loss", "ece") else comp[c].idxmax()) for c in cols}
display(Markdown("Modelo con el mejor valor medio externo en cada métrica: " +
                 "; ".join(f"**{c}**: {m}" for c, m in mejor.items()) + "."))

,auc,auc_pr,brier,log_loss,ece,f1,precision,recall,accuracy,auc_de
modelo,,,,,,,,,,
AJ-KL (original),0.7874,0.5642,0.1613,0.4829,0.0117,0.5846,0.4793,0.7504,0.6983,0.0029
LightGBM (estado del arte),0.7865,0.5630,0.1616,0.4837,0.0108,0.5826,0.4633,0.7878,0.6807,0.0029
CatBoost (estado del arte),0.7860,0.5623,0.1617,0.4844,0.0089,0.5848,0.4706,0.7724,0.6898,0.0026
XGBoost,0.7867,0.5632,0.1615,0.4835,0.0115,0.5843,0.4692,0.7754,0.6880,0.0027
Random Forest,0.7852,0.5602,0.1619,0.4846,0.0086,0.5838,0.4795,0.7470,0.6987,0.0026
SVM,0.7844,0.5573,0.1634,0.4912,0.0298,0.5828,0.4715,0.7634,0.6909,0.0023
Regresión logística,0.7809,0.5503,0.1639,0.4904,0.0214,0.5803,0.4639,0.7751,0.6829,0.0030
KNN,0.7751,0.5416,0.1661,0.4952,0.0273,0.5765,0.4565,0.7826,0.6748,0.0029
Decision Tree,0.7736,0.5364,0.1982,0.5743,0.1647,0.5722,0.4498,0.7891,0.6661,0.0032


Modelo con el mejor valor medio externo en cada métrica: **auc**: AJ-KL (original); **auc_pr**: AJ-KL (original); **brier**: AJ-KL (original); **log_loss**: AJ-KL (original); **ece**: Random Forest; **f1**: CatBoost (estado del arte); **precision**: Random Forest; **recall**: Decision Tree; **accuracy**: Random Forest.

### Tabla resumen

La guía (sección 6.3) pide, para el modelo original y los benchmarks, la media ± desviación estándar del bucle
externo, un intervalo de confianza bootstrap y los tiempos de entrenamiento e inferencia. El intervalo es del AUC
medio de los 48 meses del bucle externo con el bootstrap estacionario {cite}`politis1994`, que remuestrea bloques
de meses consecutivos y así respeta su dependencia. Los tiempos son los de los modelos de producción (entrenados
con las 622 mil filas; k-NN con su muestra de 40 mil): el del AJ-KL suma el ajuste de sus nueve modelos base y el
del combinador, y su inferencia incluye las predicciones de los nueve modelos base.

In [3]:
import estadistica as E
import experimento as X
from sklearn.metrics import roc_auc_score

mensual = mens.pivot(index="bloque", columns="modelo", values="auc")
for m_ in ["LightGBM", "CatBoost"]:
    serie = {}
    for pl in ["k1", "k2", "k3", "k4"]:
        _, Vk = X.cargar_pliegue(pl)
        p_ = joblib.load(f"../data/resultados/sota/{m_}_{pl}.joblib")["pred"]
        d_ = pd.DataFrame({"y": Vk["interrupcion_t1"].to_numpy(),
                           "mes": pd.to_datetime(Vk["semana_t1"]).dt.to_period("M").astype(str).to_numpy(),
                           "s": pd.Series(p_["s"], index=p_["fila"]).reindex(Vk["fila"]).to_numpy()})
        serie.update({mes: roc_auc_score(g["y"], g["s"]) for mes, g in d_.groupby("mes")})
    mensual[m_] = pd.Series(serie)
art = pd.read_csv("../data/resultados/artefactos_resumen.csv").set_index("modelo")
base9 = list(sel["modelo"]) + ["LightGBM", "CatBoost"]
t_ajkl = art.loc[base9, "segundos_fit"].sum() + joblib.load("../data/resultados/ajkl/final.joblib")["resumen"]["t_refit"]
equiv = {"AJ-KL (original)": ([c for c in mensual.columns if c.startswith("AJ-KL")][0], "AJ-KL (original)"),
         "LightGBM (estado del arte)": ("LightGBM", "LightGBM"), "CatBoost (estado del arte)": ("CatBoost", "CatBoost")}
equiv.update({r.modelo: (r.combinacion, r.modelo) for r in sel.itertuples()})
filas_r = []
for nombre in comp.index:
    col_m, col_a = equiv[nombre]
    ic = E.ic_bootstrap_estacionario(mensual[col_m].dropna().sort_index().to_numpy(), B=2000)
    filas_r.append({"modelo": nombre, "AUC externo (media ± DE, 4 años)": f"{comp.loc[nombre, 'auc']:.4f} ± {comp.loc[nombre, 'auc_de']:.4f}",
                    "AUC mensual medio": ic["estimado"], "IC 95 % inf.": ic["ic_inf"], "IC 95 % sup.": ic["ic_sup"],
                    "entrenamiento (s)": t_ajkl if nombre.startswith("AJ-KL") else art.loc[col_a, "segundos_fit"],
                    "inferencia (ms / 1000 filas)": art.loc[col_a, "ms_por_1000"]})
resumen_tabla = pd.DataFrame(filas_r).set_index("modelo")
display(resumen_tabla.round(4))
display(Markdown(
    f"Los intervalos se solapan: con 48 meses, el bootstrap del AUC medio de cada modelo tiene un ancho de unas "
    f"{(resumen_tabla['IC 95 % sup.'] - resumen_tabla['IC 95 % inf.']).median():.4f} unidades, mayor que la distancia "
    "entre los cuatro primeros modelos. Por eso la comparación no se hace con intervalos marginales sino con pruebas "
    "pareadas mes a mes (secciones 6.4 a 6.7), que eliminan la variación común entre meses. El AJ-KL es el más lento "
    f"en entrenar ({t_ajkl:.0f} s, porque ajusta los nueve modelos base) y el "
    f"{int((resumen_tabla['inferencia (ms / 1000 filas)'] >= resumen_tabla.loc['AJ-KL (original)', 'inferencia (ms / 1000 filas)']).sum())}.º "
    f"más lento en predecir ({resumen_tabla.loc['AJ-KL (original)', 'inferencia (ms / 1000 filas)']:.0f} ms por cada mil "
    "filas), un costo despreciable para una alerta semanal."))

,"AUC externo (media ± DE, 4 años)",AUC mensual medio,IC 95 % inf.,IC 95 % sup.,entrenamiento (s),inferencia (ms / 1000 filas)
modelo,,,,,,
AJ-KL (original),0.7874 ± 0.0029,0.7872,0.7842,0.7903,441.8564,71.3058
LightGBM (estado del arte),0.7865 ± 0.0029,0.7858,0.7827,0.7888,6.6204,12.9869
CatBoost (estado del arte),0.7860 ± 0.0026,0.7852,0.7822,0.7881,9.3642,0.8307
XGBoost,0.7867 ± 0.0027,0.7865,0.7834,0.7895,9.2738,3.0418
Random Forest,0.7852 ± 0.0026,0.7851,0.7820,0.7882,58.1029,7.6656
SVM,0.7844 ± 0.0023,0.7842,0.7811,0.7872,276.0990,5.1757
Regresión logística,0.7809 ± 0.0030,0.7808,0.7777,0.7839,10.2426,1.2305
KNN,0.7751 ± 0.0029,0.7748,0.7716,0.7782,0.1973,74.7415
Decision Tree,0.7736 ± 0.0032,0.7737,0.7704,0.7770,12.3434,0.8859


Los intervalos se solapan: con 48 meses, el bootstrap del AUC medio de cada modelo tiene un ancho de unas 0.0062 unidades, mayor que la distancia entre los cuatro primeros modelos. Por eso la comparación no se hace con intervalos marginales sino con pruebas pareadas mes a mes (secciones 6.4 a 6.7), que eliminan la variación común entre meses. El AJ-KL es el más lento en entrenar (442 s, porque ajusta los nueve modelos base) y el 2.º más lento en predecir (71 ms por cada mil filas), un costo despreciable para una alerta semanal.

## 6.3 Supuestos de cada prueba

In [4]:
t = mens.pivot(index="bloque", columns="modelo", values="auc")
ORIG = [c for c in t.columns if c.startswith("AJ-KL")][0]
filas = []
for b in sel["combinacion"]:
    d = (t[ORIG] - t[b]).to_numpy()
    filas.append({"benchmark": corto(b), "media ΔAUC": d.mean(), "asimetría": stats.skew(d),
                  "Shapiro-Wilk p": stats.shapiro(d).pvalue, "autocorrelación rezago 1": pd.Series(d).autocorr(1)})
sup = pd.DataFrame(filas).set_index("benchmark")
display(sup.round(4))
display(Markdown(
    "Wilcoxon supone diferencias simétricas alrededor de la mediana; la asimetría de las diferencias mensuales es "
    f"pequeña (entre {sup['asimetría'].min():.2f} y {sup['asimetría'].max():.2f}). La normalidad no es necesaria para "
    "Wilcoxon, pero sí para la t de Nadeau-Bengio, por eso esta última se usa solo como prueba complementaria. La "
    "autocorrelación de rezago 1 de las diferencias mensuales indica cuánto se parecen meses consecutivos: cuando no "
    "es despreciable, los p de Wilcoxon son algo optimistas y la evidencia debe leerse junto con las pruebas que "
    "respetan la dependencia (permutación por pliegue, DeLong con bloques de semana, Diebold-Mariano con varianza HAC)."))

,media ΔAUC,asimetría,Shapiro-Wilk p,autocorrelación rezago 1
benchmark,,,,
XGBoost,0.0007,1.4006,0.0007,0.0350
Random Forest,0.0021,0.6856,0.1110,-0.0030
SVM,0.0030,0.3035,0.0493,0.2507
Regresión logística,0.0064,0.3786,0.2240,-0.1749
KNN,0.0124,-0.6677,0.1232,0.1920
Decision Tree,0.0135,0.4978,0.5529,0.0319
Naive Bayes,0.0188,0.9225,0.0198,0.2294


Wilcoxon supone diferencias simétricas alrededor de la mediana; la asimetría de las diferencias mensuales es pequeña (entre -0.67 y 1.40). La normalidad no es necesaria para Wilcoxon, pero sí para la t de Nadeau-Bengio, por eso esta última se usa solo como prueba complementaria. La autocorrelación de rezago 1 de las diferencias mensuales indica cuánto se parecen meses consecutivos: cuando no es despreciable, los p de Wilcoxon son algo optimistas y la evidencia debe leerse junto con las pruebas que respetan la dependencia (permutación por pliegue, DeLong con bloques de semana, Diebold-Mariano con varianza HAC).

## 6.4 Tabla de pruebas de hipótesis

In [5]:
mostrar = tabla.copy()
mostrar["conclusión"] = np.where(mostrar["rechaza_h0"], "Rechaza H0", "No rechaza H0")
display(mostrar[["prueba", "comparacion", "h0", "supuestos", "estadistico", "p_crudo", "p_ajustado", "efecto", "conclusión"]]
        .round({"estadistico": 3}))
n_comp = len(tabla)
display(Markdown(f"Se reportan {n_comp} contrastes en total: 1 de Friedman, 7 de Nemenyi, 7 de Wilcoxon, 7 de DeLong, "
                 "2 MCS y 7 de Diebold-Mariano, además de las 7 t de Nadeau-Bengio y las 28 pruebas bayesianas de las "
                 "tablas siguientes."))

,prueba,comparacion,h0,supuestos,estadistico,p_crudo,p_ajustado,efecto,conclusión
0,Friedman (Iman-Davenport) + permutación por pliegue,8 modelos,los 8 modelos tienen el mismo AUC mensual (mismos rangos),bloques mensuales; dependencia dentro del pliegue tratada con permutación restringida,620.659,9.999000e-05,9.999000e-05,rango AJ-KL (original) 1.27 (CD = 1.52),Rechaza H0
1,Nemenyi (post hoc),AJ-KL (original) vs XGBoost,rangos medios iguales,post hoc de Friedman; ajuste incluido en q de Tukey,0.813,7.352451e-01,7.352451e-01,Δrango = 0.81,No rechaza H0
2,Nemenyi (post hoc),AJ-KL (original) vs Random Forest,rangos medios iguales,post hoc de Friedman; ajuste incluido en q de Tukey,1.833,6.005108e-03,6.005108e-03,Δrango = 1.83,Rechaza H0
3,Nemenyi (post hoc),AJ-KL (original) vs SVM,rangos medios iguales,post hoc de Friedman; ajuste incluido en q de Tukey,2.333,8.323043e-05,8.323043e-05,Δrango = 2.33,Rechaza H0
4,Nemenyi (post hoc),AJ-KL (original) vs Regresión logística,rangos medios iguales,post hoc de Friedman; ajuste incluido en q de Tukey,3.729,2.453260e-12,2.453260e-12,Δrango = 3.73,Rechaza H0
5,Nemenyi (post hoc),AJ-KL (original) vs KNN,rangos medios iguales,post hoc de Friedman; ajuste incluido en q de Tukey,5.146,0.000000e+00,0.000000e+00,Δrango = 5.15,Rechaza H0
6,Nemenyi (post hoc),AJ-KL (original) vs Decision Tree,rangos medios iguales,post hoc de Friedman; ajuste incluido en q de Tukey,5.333,0.000000e+00,0.000000e+00,Δrango = 5.33,Rechaza H0
7,Nemenyi (post hoc),AJ-KL (original) vs Naive Bayes,rangos medios iguales,post hoc de Friedman; ajuste incluido en q de Tukey,6.646,0.000000e+00,0.000000e+00,Δrango = 6.65,Rechaza H0
8,Wilcoxon pareado (Holm),AJ-KL (original) vs XGBoost,mediana de ΔAUC mensual = 0,diferencias simétricas; 48 meses,114.000,1.119195e-07,1.119195e-07,Cliff pareado 0.62 (grande); d_z 0.79,Rechaza H0
9,Wilcoxon pareado (Holm),AJ-KL (original) vs Random Forest,mediana de ΔAUC mensual = 0,diferencias simétricas; 48 meses,1.000,1.421085e-14,4.973799e-14,Cliff pareado 0.96 (grande); d_z 1.74,Rechaza H0


Se reportan 31 contrastes en total: 1 de Friedman, 7 de Nemenyi, 7 de Wilcoxon, 7 de DeLong, 2 MCS y 7 de Diebold-Mariano, además de las 7 t de Nadeau-Bengio y las 28 pruebas bayesianas de las tablas siguientes.

## 6.5 Diagrama de diferencia crítica

In [6]:
PL.diagrama_cd(pd.Series({corto(k): v for k, v in res["friedman"]["rangos"].items()}), res["friedman"]["CD"],
               "Friedman-Nemenyi: rango medio del AUC en 48 meses del bucle externo (1 = mejor)").show()

## 6.6 Tamaños del efecto e intervalos

Se reportan dos versiones del delta de Cliff porque responden preguntas distintas. El **delta de Cliff
de dos muestras** {cite}`cliff1993` compara la distribución de los 48 AUC mensuales del AJ-KL con la del
benchmark como si fueran muestras independientes; la variación entre meses (choques, estacionalidad) es
mucho mayor que la diferencia entre modelos y lo vuelve pequeño. La **dominancia pareada** es la media de
signo(ΔAUC) mes a mes: mide qué tan consistente es la ventaja cuando se compara en el mismo mes. La guía
acepta también la d de Cohen pareada, $d_z$ = media / desviación estándar de las diferencias.

In [7]:
w = W[W.metrica == "auc"].copy()
w["benchmark"] = w["benchmark"].map(corto)
display(w[["benchmark", "media_dif", "ic_inf", "ic_sup", "meses_gana", "p_holm", "cliff", "mag_cliff",
           "cliff_pareado", "mag_cliff_pareado", "d_cohen", "mag_d", "bayes_p_izq", "bayes_p_rope", "bayes_p_der"]]
        .round(4).set_index("benchmark"))
otros = W[W.metrica != "auc"].assign(benchmark=lambda d: d["benchmark"].map(corto))
display(otros.pivot(index="benchmark", columns="metrica", values="meses_gana").rename(columns=lambda c: f"meses que gana ({c})"))

,media_dif,ic_inf,ic_sup,meses_gana,p_holm,cliff,mag_cliff,cliff_pareado,mag_cliff_pareado,d_cohen,mag_d,bayes_p_izq,bayes_p_rope,bayes_p_der
benchmark,,,,,,,,,,,,,,
XGBoost,0.0007,0.0005,0.0010,39,0.0,0.0451,despreciable,0.6250,grande,0.7864,mediano,0.0,1.0,0.0
Random Forest,0.0021,0.0018,0.0025,47,0.0,0.1241,despreciable,0.9583,grande,1.7431,grande,0.0,1.0,0.0
SVM,0.0030,0.0025,0.0036,45,0.0,0.1710,pequeño,0.8750,grande,1.6807,grande,0.0,1.0,0.0
Regresión logística,0.0064,0.0059,0.0070,48,0.0,0.3681,mediano,1.0000,grande,3.3068,grande,0.0,1.0,0.0
KNN,0.0124,0.0117,0.0131,48,0.0,0.6285,grande,1.0000,grande,5.4565,grande,0.0,0.0,1.0
Decision Tree,0.0135,0.0128,0.0142,48,0.0,0.6701,grande,1.0000,grande,5.5142,grande,0.0,0.0,1.0
Naive Bayes,0.0188,0.0176,0.0202,48,0.0,0.8064,grande,1.0000,grande,5.2026,grande,0.0,0.0,1.0


metrica,meses que gana (auc_pr),meses que gana (brier),meses que gana (log_loss)
benchmark,,,
Decision Tree,48,48,48
KNN,48,48,48
Naive Bayes,48,48,48
Random Forest,42,44,42
Regresión logística,48,48,48
SVM,47,47,47
XGBoost,29,36,38


La prueba bayesiana de rangos con signo {cite}`benavoli2014signedrank,benavoli2017bayesian` usa una región
de equivalencia práctica (ROPE) de ±0,01 en AUC, la frontera entre diferencias «triviales» y «pequeñas» de
la escala del curso. Las columnas `bayes_p_*` son las probabilidades posteriores de que el benchmark sea
mejor, de que la diferencia caiga en la ROPE y de que el AJ-KL sea mejor.

## 6.7 Diferencias en la prueba 2025 (intervalos BCa por bloques de semana)

In [8]:
d = Dl.copy()
d["benchmark"] = d["benchmark"].map(corto)
for m_ in ["auc", "auc_pr", "brier", "log_loss", "ece"]:
    d[m_] = [f"{a:+.4f} [{b:+.4f}; {c:+.4f}]" for a, b, c in zip(d[f"{m_}_dif"], d[f"{m_}_ic_inf"], d[f"{m_}_ic_sup"])]
display(d.set_index("benchmark")[["auc", "auc_pr", "brier", "log_loss", "ece", "p_delong_iid", "p_delong_bloques", "p_holm"]])
display(Markdown("Las diferencias son AJ-KL − benchmark: positivas son mejores en AUC y AUC-PR; negativas son mejores en "
                 "Brier, log-loss y ECE."))

,auc,auc_pr,brier,log_loss,ece,p_delong_iid,p_delong_bloques,p_holm
benchmark,,,,,,,,
XGBoost,+0.0011 [+0.0007; +0.0014],+0.0016 [+0.0003; +0.0028],-0.0003 [-0.0004; -0.0002],-0.0010 [-0.0013; -0.0007],-0.0002 [-0.0021; +0.0015],1.862235e-10,1.771746e-09,3.543492e-09
Random Forest,+0.0022 [+0.0014; +0.0035],+0.0041 [+0.0023; +0.0060],-0.0008 [-0.0011; -0.0006],-0.0026 [-0.0036; -0.0021],-0.0060 [-0.0082; -0.0043],1.276056e-20,2.490528e-05,2.490528e-05
SVM,+0.0030 [+0.0022; +0.0041],+0.0080 [+0.0053; +0.0111],-0.0021 [-0.0026; -0.0018],-0.0096 [-0.0107; -0.0085],-0.0241 [-0.0280; -0.0206],1.549067e-18,2.494770e-10,7.484311e-10
Regresión logística,+0.0059 [+0.0050; +0.0069],+0.0159 [+0.0134; +0.0185],-0.0025 [-0.0029; -0.0022],-0.0077 [-0.0086; -0.0069],-0.0142 [-0.0182; -0.0109],4.990335e-51,9.926975e-35,3.970790e-34
KNN,+0.0134 [+0.0121; +0.0148],+0.0274 [+0.0240; +0.0306],-0.0046 [-0.0054; -0.0042],-0.0127 [-0.0142; -0.0116],-0.0146 [-0.0213; -0.0082],7.225689e-138,6.392176e-92,4.474523e-91
Decision Tree,+0.0115 [+0.0102; +0.0136],+0.0232 [+0.0201; +0.0268],-0.0384 [-0.0404; -0.0354],-0.0958 [-0.1003; -0.0903],-0.1614 [-0.1650; -0.1580],5.804349e-102,6.737067e-44,3.368534e-43
Naive Bayes,+0.0178 [+0.0162; +0.0196],+0.0508 [+0.0466; +0.0549],-0.0677 [-0.0702; -0.0657],-0.4382 [-0.4518; -0.4278],-0.1807 [-0.1880; -0.1775],8.623422e-182,6.499636e-92,4.474523e-91


Las diferencias son AJ-KL − benchmark: positivas son mejores en AUC y AUC-PR; negativas son mejores en Brier, log-loss y ECE.

In [9]:
nb = NB.assign(benchmark=lambda x: x["benchmark"].map(corto)).set_index("benchmark")
display(nb.round(4))
dm = DM.assign(benchmark=lambda x: x["benchmark"].map(corto))
display(dm.pivot(index="benchmark", columns="perdida", values="p_holm").round(4).rename(columns=lambda c: f"p Holm DM ({c})"))

,t,gl,p,media,pliegues_gana,p_holm
benchmark,,,,,,
XGBoost,1.9024,4,0.1299,0.0008,5,0.1299
Random Forest,3.4790,4,0.0254,0.0022,5,0.0761
SVM,3.3262,4,0.0292,0.0030,5,0.0761
Regresión logística,5.1997,4,0.0065,0.0064,5,0.0261
KNN,7.9746,4,0.0013,0.0126,5,0.0094
Decision Tree,7.1174,4,0.0021,0.0133,5,0.0124
Naive Bayes,5.7349,4,0.0046,0.0187,5,0.0229


perdida,p Holm DM (brier),p Holm DM (log_loss)
benchmark,,
Decision Tree,0.0,0.0
KNN,0.0,0.0
Naive Bayes,0.0,0.0
Random Forest,0.0,0.0
Regresión logística,0.0,0.0
SVM,0.0,0.0
XGBoost,0.0,0.0


## 6.8 Comparación con el estado del arte reciente

LightGBM {cite}`ke2017lightgbm` y CatBoost {cite}`prokhorenkova2018catboost` son los GBDT que la literatura
reciente sobre datos tabulares sigue señalando como referencia {cite}`shwartzziv2022tabular,borisov2024tabular`.
No se vieron en clase; se ajustaron con exactamente el mismo protocolo que los benchmarks (mismos pliegues,
misma muestra $S_k$, TPE con 24 evaluaciones y semilla 42). La tabla resume, para cada uno, el AUC medio del
bucle externo, la diferencia en 2025 con DeLong por bloques de semana y los meses del bucle externo en que el
AJ-KL gana.

In [10]:
import os
if os.path.exists(V + "comparadores_sota.parquet"):
    display(pd.read_parquet(V + "comparadores_sota.parquet").round(5))

,comparador,auc_ext_orig,auc_ext_comp,meses_gana,p_wilcoxon,delta_2025,ic_inf,ic_sup,p_delong_bloques,brier_dif,log_loss_dif,ece_dif,p_wilcoxon_holm,p_delong_holm
0,LightGBM,0.78721,0.78636,40,0.0,0.00040,0.00003,0.00078,0.03384,-0.00024,-0.00151,-0.00557,0.0,0.06769
1,CatBoost,0.78721,0.78589,44,0.0,0.00028,-0.00018,0.00069,0.22158,-0.00005,-0.00073,-0.00000,0.0,0.22158


## 6.9 Comparación con información igual: modelos de producción en 2025

En la validación anidada, el combinador del AJ-KL ve las predicciones fuera de muestra de los bloques
externos anteriores, más filas etiquetadas que la muestra $S_k$ de 40 mil filas de los benchmarks (aunque
ninguna posterior a su corte de entrenamiento). La sección 5.8 muestra que reducir ese historial a 40 mil
filas no cambia el resultado. Una segunda forma de igualar la información es la de producción (Bloque B):
todos los modelos se reentrenan con las 622.244 filas de entrenamiento, y el AJ-KL combina esos nueve
artefactos con el combinador del pliegue final, entrenado con predicciones de semanas que los benchmarks
también usaron para entrenar. Cada artefacto se evalúa una sola vez en las 103.466 filas de 2025.

In [11]:
if os.path.exists(V + "produccion_2025.parquet"):
    pr = pd.read_parquet(V + "produccion_2025.parquet").assign(benchmark=lambda x: x["benchmark"].map(corto))
    for m_ in ["auc", "auc_pr", "brier", "log_loss", "ece"]:
        pr[m_] = [f"{a:+.4f} [{b:+.4f}; {c:+.4f}]" for a, b, c in zip(pr[f"{m_}_dif"], pr[f"{m_}_ic_inf"], pr[f"{m_}_ic_sup"])]
    display(pr.set_index("benchmark")[["auc_orig", "auc_bench", "auc", "p_delong_holm", "mcnemar_orig_acierta",
                                        "mcnemar_bench_acierta", "p_mcnemar_holm", "brier", "log_loss", "ece"]].round(5))
    art = pd.read_csv("../data/resultados/artefactos_resumen.csv")
    display(art[["modelo", "auc_2025", "auc_pr_2025", "brier_2025", "log_loss_2025", "ece_2025", "f1_2025", "mb", "ms_por_1000"]]
            .sort_values("auc_2025", ascending=False).round(4).set_index("modelo"))

,auc_orig,auc_bench,auc,p_delong_holm,mcnemar_orig_acierta,mcnemar_bench_acierta,p_mcnemar_holm,brier,log_loss,ece
benchmark,,,,,,,,,,
XGBoost,0.80072,0.79994,+0.0008 [+0.0004; +0.0012],0.00043,2119,924,0.0,-0.0002 [-0.0004; -0.0001],-0.0008 [-0.0011; -0.0005],+0.0004 [-0.0011; +0.0015]
Random Forest,0.80072,0.79827,+0.0025 [+0.0015; +0.0041],0.00017,1277,1596,0.0,-0.0008 [-0.0012; -0.0006],-0.0029 [-0.0040; -0.0023],-0.0040 [-0.0058; -0.0019]
SVM,0.80072,0.79847,+0.0022 [+0.0014; +0.0034],0.00003,2997,1698,0.0,-0.0015 [-0.0020; -0.0012],-0.0078 [-0.0088; -0.0069],-0.0156 [-0.0214; -0.0108]
Regresión logística,0.80072,0.79343,+0.0073 [+0.0064; +0.0083],0.00000,5041,1955,0.0,-0.0028 [-0.0032; -0.0025],-0.0083 [-0.0092; -0.0075],-0.0067 [-0.0116; -0.0030]
KNN,0.80072,0.78510,+0.0156 [+0.0144; +0.0173],0.00000,4186,3163,0.0,-0.0052 [-0.0061; -0.0047],-0.0142 [-0.0162; -0.0130],-0.0092 [-0.0193; -0.0017]
Decision Tree,0.80072,0.79270,+0.0080 [+0.0067; +0.0098],0.00000,4553,2430,0.0,-0.0379 [-0.0398; -0.0348],-0.0920 [-0.0968; -0.0847],-0.1611 [-0.1632; -0.1567]
Naive Bayes,0.80072,0.78017,+0.0206 [+0.0190; +0.0226],0.00000,4274,3289,0.0,-0.0675 [-0.0698; -0.0654],-0.4481 [-0.4614; -0.4373],-0.1769 [-0.1851; -0.1718]
LightGBM,0.80072,0.80108,-0.0004 [-0.0007; -0.0000],0.07797,1315,790,0.0,-0.0000 [-0.0001; +0.0001],-0.0010 [-0.0013; -0.0008],-0.0046 [-0.0073; -0.0026]
CatBoost,0.80072,0.80113,-0.0004 [-0.0009; -0.0000],0.07797,1370,946,0.0,+0.0002 [+0.0001; +0.0003],-0.0001 [-0.0004; +0.0002],+0.0004 [-0.0018; +0.0018]


,auc_2025,auc_pr_2025,brier_2025,log_loss_2025,ece_2025,f1_2025,mb,ms_por_1000
modelo,,,,,,,,
CatBoost,0.8011,0.5602,0.1507,0.4554,0.0107,0.5786,0.1848,0.8307
LightGBM,0.8011,0.5591,0.1509,0.4563,0.0157,0.5786,0.5098,12.9869
AJ-KL (original),0.8007,0.5580,0.1509,0.4553,0.0111,0.5780,11.1959,71.3058
XGBoost,0.7999,0.5566,0.1511,0.4561,0.0107,0.5778,0.1339,3.0418
SVM,0.7985,0.5523,0.1524,0.4631,0.0267,0.5759,0.7368,5.1757
Random Forest,0.7983,0.5545,0.1517,0.4582,0.0151,0.5762,1.6803,7.6656
Regresión logística,0.7934,0.5386,0.1537,0.4636,0.0178,0.5696,0.0160,1.2305
Decision Tree,0.7927,0.5415,0.1888,0.5473,0.1722,0.5701,0.0304,0.8859
KNN,0.7851,0.5264,0.1561,0.4695,0.0203,0.5656,6.8484,74.7415


## 6.10 Veredicto

In [12]:
rm = json.loads(open("../data/dashboard/resumen_modelos.json").read())
display(Markdown(rm["veredicto"]))

Superior según la definición operativa de la guía (sección 6.1). (1) AUC externo medio del AJ-KL 0,7874 frente a 0,7867 de XGBoost, el mejor de los siete benchmarks. (2) 5 pruebas distintas rechazan la igualdad frente a XGBoost con p ajustado < 0,05 (0,10 en el MCS): DeLong, Diebold-Mariano, Friedman, Model Confidence Set, Wilcoxon pareado; no rechazan: Nemenyi. (3) Tamaño del efecto: d de Cohen pareada 0,79, dominancia pareada 0,62 (gana 39 de 48 meses) y delta de Cliff de dos muestras 0,05 (despreciable). Con los modelos de producción (622 mil filas, prueba 2025) la diferencia frente a XGBoost es ΔAUC 0,0008 [IC BCa 0,0004; 0,0012] (DeLong con bloques de semana, p Holm < 0,001). Frente a los comparadores recientes: LightGBM ΔAUC 2025 0,0004 (p Holm DeLong 0,068), gana 40 de 48 meses; CatBoost ΔAUC 2025 0,0003 (p Holm DeLong 0,222), gana 44 de 48 meses. En unidades de AUC la ventaja media es 0,0007 (escala del curso: trivial).